# Port Scanner

This notebook validates targets, resolves hostnames, scans a list or range of ports, and reports which ports are open.

Note: Only scan systems you own or have explicit permission to test.


In [ ]:
import socket
import threading
from datetime import datetime

COMMON_SERVICES = {
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    80: "HTTP",
    110: "POP3",
    143: "IMAP",
    443: "HTTPS",
    587: "SMTP Submission",
    8080: "HTTP Alternate",
}

def validate_target(target):
    if not isinstance(target, str) or not target.strip():
        raise ValueError("Target must be a non-empty string.")
    return target.strip()

def normalize_ports(port_input):
    if isinstance(port_input, int):
        return [port_input]

    if isinstance(port_input, str):
        values = []
        for part in port_input.split(","):
            part = part.strip()
            if not part:
                continue
            if "-" in part:
                start, end = map(int, part.split("-", 1))
                values.extend(range(start, end + 1))
            else:
                values.append(int(part))
        return values

    if isinstance(port_input, (list, tuple, set)):
        values = []
        for port in port_input:
            if isinstance(port, str) and "-" in port:
                start, end = map(int, port.split("-", 1))
                values.extend(range(start, end + 1))
            else:
                values.append(int(port))
        return values

    raise ValueError("Ports must be an int, a comma-separated string, or a list/tuple/set of ints.")

def scan_port(target_ip, port, timeout_seconds=0.5):
    scanner = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    scanner.settimeout(timeout_seconds)

    try:
        result = scanner.connect_ex((target_ip, port))
        return "open" if result == 0 else "closed"
    except socket.timeout:
        return "closed"
    except OSError as error:
        return f"error: {error}"
    finally:
        scanner.close()

def run_port_scan(target="127.0.0.1", ports=None, timeout_seconds=0.5):
    if ports is None:
        ports = [21, 22, 80, 443]

    target = validate_target(target)
    normalized_ports = normalize_ports(ports)

    for port in normalized_ports:
        if not 1 <= port <= 65535:
            raise ValueError(f"Invalid port '{port}'. Ports must be between 1 and 65535.")

    if not isinstance(timeout_seconds, (int, float)):
        raise ValueError("Timeout must be a number.")
    if timeout_seconds <= 0:
        raise ValueError("Timeout must be greater than 0.")

    try:
        target_ip = socket.gethostbyname(target)
    except socket.gaierror as exc:
        raise ValueError(f"Could not resolve target hostname: {target}") from exc

    start_time = datetime.now()
    results = []
    lock = threading.Lock()

    def worker(port):
        status = scan_port(target_ip, port, timeout_seconds)
        service_name = COMMON_SERVICES.get(port, "Unknown")
        with lock:
            results.append({
                "port": port,
                "service": service_name,
                "status": status,
            })

    threads = [threading.Thread(target=worker, args=(port,)) for port in sorted(set(normalized_ports))]

    print(f"Starting scan")
    print(f"Target: {target}")
    print(f"Resolved IP: {target_ip}")
    print(f"Ports: {sorted(set(normalized_ports))}")
    print(f"Timeout: {timeout_seconds} seconds")
    print("-" * 50)

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    results.sort(key=lambda item: item["port"])
    open_ports = [item["port"] for item in results if item["status"] == "open"]
    closed_ports = [item["port"] for item in results if item["status"] == "closed"]
    error_ports = [item for item in results if item["status"].startswith("error")]

    for item in results:
        label = item["status"]
        print(f"Port {item['port']:<5} {item['service']:<18} {label}")

    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()

    print("-" * 50)
    print(f"Open ports: {open_ports}")
    print(f"Closed ports: {closed_ports}")
    if error_ports:
        print(f"Errors: {error_ports}")
    print(f"Scan completed in {duration:.3f} seconds")

    return {
        "target": target,
        "target_ip": target_ip,
        "open_ports": open_ports,
        "closed_ports": closed_ports,
        "error_ports": error_ports,
        "results": results,
        "duration": duration,
    }


In [ ]:
localhost_results = run_port_scan(
    target="127.0.0.1",
    ports="21,22,80,443,8080",
    timeout_seconds=0.5,
)


In [ ]:
scanme_results = run_port_scan(
    target="scanme.nmap.org",
    ports=[22, 80, 443],
    timeout_seconds=1.0,
)


In [ ]:
invalid_tests = [
    {"target": "", "ports": [80], "timeout_seconds": 0.5},
    {"target": "example.com", "ports": [80], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": [0, 80], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": ["80"], "timeout_seconds": 0.5},
    {"target": "127.0.0.1", "ports": [80], "timeout_seconds": -1},
]

for test in invalid_tests:
    try:
        run_port_scan(**test)
    except ValueError as error:
        print(f"Handled invalid input: {error}")
